# Initialize

In [ ]:
import numpy as np
import pandas as pd
import logging
import time 
import cohere
import json
import glob
import re
# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# ignore warnings
import warnings
warnings.filterwarnings("ignore")

from openai import OpenAI
openrouter_api = "your_openrouter_api_key_here"
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=openrouter_api,
)

cohere_apiKey = 'your_cohere_api_key_here'
co = cohere.ClientV2(cohere_apiKey)


# matbench_steels

In [ ]:
with open('steels_yield.json') as json_data:
    data = json.load(json_data)
    df = pd.DataFrame(data['data'],columns=data['columns']).sample(frac=1, random_state=42).reset_index(drop=True)

# Custom function to parse the composition strings
def custom_parse_composition(comp_str):
    # Regular expression to match elements and their amounts
    pattern = re.compile(r'([A-Z][a-z]*)([0-9\.]+)')
    return dict(pattern.findall(comp_str))
# Apply the custom parsing function
element_data = df['composition'].apply(custom_parse_composition)
# Convert the list of dictionaries into a DataFrame
element_df = pd.DataFrame(list(element_data))
# Replace missing values with 0
element_df = element_df.fillna(0)
# Convert string numbers to float
element_df = element_df.apply(pd.to_numeric)
# Combine the element DataFrame with the original DataFrame
df = pd.concat([df, element_df], axis=1)

# Systematically generate a column with "column_name: value" format for each row except the columns 'composition' and 'yield strength'
# Define the columns to be included in the formatted string
def generate_formatted_column(row):
    formatted_str = ', '.join([f"{col}: {row[col]}" for col in df.columns if col not in ['composition', 'yield strength']])
    return formatted_str

# Apply this function to each row to create a new column
df['Formatted_Parameters'] = df.apply(generate_formatted_column, axis=1)


def Generate_report(Formatted_Parameters, Model = "anthropic/claude-3.7-sonnet", Temperature = 0.0, sleep = 0.5):
    response = client.chat.completions.create(
      model= Model,
      messages=[{"role": "system", "content": 'Here is the elemental composition of a steel sample. Could you generate a brief experimental-style report that outlines how this composition could inform an estimation of its yield strength, including relevant metallurgical considerations or trends.'},
                {"role": "user", "content": Formatted_Parameters}

                
                ],
      temperature=Temperature
    )
    time.sleep(sleep)
    return response.choices[0].message.content

# Generate the report for the dataframe
df['Report'] = df['Formatted_Parameters'].apply(Generate_report, Model = "anthropic/claude-3.7-sonnet", Temperature = 0.0, sleep = 0.5)

df['Report with output'] = df.apply(lambda row: f"Report: {row['Report']}" + f" \n\n**The Actual Yield Strength is {round(row['yield strength'], 2)} MPa**", axis=1)

df['Formatted_Parameters with output'] = df.apply(lambda row: f"Parameters: {row['Formatted_Parameters']}" + f" \n\n**The Actual Yield Strength is {round(row['yield strength'], 2)} MPa**", axis=1)

df.to_csv('matbench_steels.csv', index=False)


# P3HT_dataset

In [ ]:
df = pd.read_csv("P3HT_dataset_original.csv")

# Systematically generate a column with "column_name: value" format for each row except the columns 'composition' and 'yield strength'
# Define the columns to be included in the formatted string
def generate_formatted_column(row):
    formatted_str = ', '.join([f"{col}: {row[col]}" for col in df.columns if col not in ['composition', 'yield strength']])
    return formatted_str

# Apply this function to each row to create a new column
df['Formatted_Parameters'] = df.apply(generate_formatted_column, axis=1)

print(df['Formatted_Parameters'][0])

def Generate_report(Formatted_Parameters, Model = "anthropic/claude-3.7-sonnet", Temperature = 0.0, sleep = 0.5):
    response = client.chat.completions.create(
      model= Model,
      messages=[{"role": "system", "content": "You are provided with the composition of a thin film composite containing poly(3-hexylthiophene) (P3HT) and carbon nanotubes (CNTs). "
                                              "The CNTs include types such as long single-walled (D1), short single-walled (D2), double-walled (D6), and multi-walled (D8). "
                                              "Please write a concise experimental-style report that describes how this composition might influence the resulting film's electrical conductivity. "
                                              "Base your explanation on principles of conductive network formation, percolation, CNT dispersion, and polymer-nanotube interaction. "
                                              "If relevant, mention trends like the effect of P3HT ratio or CNT type on charge transport. "},
                {"role": "user", "content": Formatted_Parameters}

                
                ],
      temperature=Temperature
    )
    time.sleep(sleep)
    return response.choices[0].message.content

# Generate the report for the dataframe
df['Report'] = df['Formatted_Parameters'].apply(Generate_report, Model = "anthropic/claude-3.7-sonnet", Temperature = 0.0, sleep = 0.5)

df['Report with output'] = df.apply(lambda row: f"Report: {row['Report']}" + f" \n\n**The Actual Conductivity is {round(row['Conductivity (measured) (S/cm)'], 2)} S/cm**", axis=1)

df['Formatted_Parameters with output'] = df.apply(lambda row: f"Parameters: {row['Formatted_Parameters']}" + f" \n\n**The Actual Conductivity is {round(row['Conductivity (measured) (S/cm)'], 2)} S/cm**", axis=1)

df.to_csv('P3HT_dataset.csv', index=False)

# Perovskite_dataset

In [ ]:
df = pd.read_csv("Perovskite_dataset_original.csv")
# Systematically generate a column with "column_name: value" format for each row except the columns 'composition' and 'yield strength'
# Define the columns to be included in the formatted string
def generate_formatted_column(row):
    formatted_str = ', '.join([f"{col}: {row[col]}" for col in df.columns if col not in ['composition', 'yield strength']])
    return formatted_str

# Apply this function to each row to create a new column
df['Formatted_Parameters'] = df.apply(generate_formatted_column, axis=1)

def Generate_report(Formatted_Parameters, Model="anthropic/claude-3.7-sonnet", Temperature=0.0, sleep=0.5):
    response = client.chat.completions.create(
        model=Model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are provided with the composition of a perovskite material made from varying ratios of CsPbI₃, FAPbI₃, and MAPbI₃. "
                    "Please write a concise experimental-style report that discusses how this composition may relate to the film’s overall properties, "
                    "with particular attention to aspects that could influence long-term stability. "
                    "Base your explanation on general structural and chemical considerations without assuming any specific outcome."
                )
            },
            {
                "role": "user",
                "content": Formatted_Parameters
            }
        ],
        temperature=Temperature
    )
    time.sleep(sleep)
    return response.choices[0].message.content

# Generate the report for the dataframe
df['Report'] = df['Formatted_Parameters'].apply(Generate_report, Model = "anthropic/claude-3.7-sonnet", Temperature = 0.0, sleep = 0.5)

df['Report with output'] = df.apply(lambda row: f"Report: {row['Report']}" + f" \n\n**The Actual Instability index is {row['Instability index']}**", axis=1)

df['Formatted_Parameters with output'] = df.apply(lambda row: f"Parameters: {row['Formatted_Parameters']}" + f" \n\n**The Actual Instability index is {row['Instability index']}**", axis=1)

df.to_csv('Perovskite_dataset.csv', index=False)

# Membrane_dataset

In [ ]:
df = pd.read_csv('Membrane_dataset_original.csv')
df.drop(columns=['Batch', 'Sample', 'Date', 'Report', 'Formatted_Parameters'], inplace=True)
# insert a index column
df.insert(0, 'Index', range(1, 1 + len(df)))

# Systematically generate a column with "column_name: value" format for each row
def generate_formatted_column(row):
    return '; '.join([f"{col}: {row[col]}" for col in row.index])

# Apply this function to each row to create a new column
df['Formatted_Parameters'] = df.apply(generate_formatted_column, axis=1)

def Generate_report(Formatted_Parameters, Model = "anthropic/claude-3.7-sonnet", Temperature = 0.0, sleep = 0.5):
    response = client.chat.completions.create(
      model= Model,
      messages=[{"role": "system", "content": 'Here I have a set of structured experimental parameters for my automated membrane synthesis via non-solvent-induced phase seperation (NIPS) using my robotic system. Index is the index number of the experiment. Auto means that whether the synthesis is automated or manual. Heating means whether the solution mixing process has heating involved. Concentration means the polymer (polysulfone) concentration in the solution (solvent is PolarClean). Heating block temperature means the temperature of the heating block during the mixing process. Mixed means whether the solution is mixed (diluted) by the robot or premixed manually. Rest time means the number days the solution is rested in a sample vial after mixing before blade casting. Coupon-to-Bath Wait Time (min) means how long the sample is waited in minutes after blade casting before immersion in the NIPS bath. Relative Humidity means the RH% of the day the sample is synthesized into membrane. Nitrogen (Side from drop) means whether the polymer solution undergoes a laminar dry nitrogen blow to remove humidity since it is drop-cast on to the coupon before blade casting until it is immersed into the NIPS bath. Nitrogen (After blade) means whether the polymer solution undergoes a laminar dry nitrgen blow after it is blade-cast until it is immersed into the NIPS bath. Can you translate that into a short experimental report? '},
                {"role": "user", "content": Formatted_Parameters}
                ],
      temperature=Temperature
    )
    time.sleep(sleep)
    return response.choices[0].message.content

reports = []  # Initialize an empty list to store each generated report

for row in df['Formatted_Parameters']:
    report = Generate_report(row)
    reports.append(report)  # Append each generated report to the list

df['Report'] = reports  # Assign the list of reports to the DataFrame

df_parameters = pd.read_csv('Membrane_dataset_original.csv')
df = pd.read_csv('Membrane_dataset_properties.csv')
data = pd.merge(df, df_parameters, on=['Auto', 'Heating', 'Concentration', 'Batch', 'Sample'])

# Group by the specified columns and calculate mean and std for the required properties
data_grouped = data.groupby(['Auto', 'Heating', 'Concentration', 'Batch', 'Sample']).agg({
    'Elastic Modulus': ['mean', 'std'],
    'Yield Strength': ['mean', 'std'],
    'Creep Strain': ['mean', 'std'],
    'Plateau Slope': ['mean', 'std'],
    'Densification Slope': ['mean', 'std'],
    'Changepoint': ['mean', 'std'],
    'Average Standard Deviation': ['mean', 'std'],
    'Coefficient of Variation': ['mean', 'std']
})

# Flatten MultiIndex columns after aggregation
data_grouped.columns = ['_'.join(col).strip() for col in data_grouped.columns.values]
data_grouped = data_grouped.reset_index()

# Merge the grouped data with df_parameters
data_grouped = pd.merge(data_grouped, df_parameters, on=['Auto', 'Heating', 'Concentration', 'Batch', 'Sample'])

data_grouped['Report with output'] = data_grouped.apply(lambda row: f"Report: {row['Report']}" + f" \n\n**The measured Elastic Modulus is {round(row['Elastic Modulus_mean'], 2)} with a standard deviation of {round(row['Elastic Modulus_std'], 2)} bar.**", axis=1)
print(data_grouped['Report with output'][0])

data_grouped['Formatted_Parameters with output'] = data_grouped.apply(lambda row: f"Parameters: {row['Formatted_Parameters']}" + f" \n\n**The measured Elastic Modulus is {round(row['Elastic Modulus_mean'], 2)} with a standard deviation of {round(row['Elastic Modulus_std'], 2)} bar.**", axis=1)

data_grouped.to_csv('Membrane_dataset.csv', index=False)
